In [6]:
import numpy as np
import torch
import sys

from torch.utils.data import DataLoader

sys.path.append('..')

from models.SRCNN.model import SRCNN
from datasets.super_res_dataset_fourier import SuperResDataset

from evaluation.evaluate import evaluate_model

In [8]:
test_dataset = SuperResDataset(
    hr_files=["../data/100/window_2023.npy"],
    downsample_factor=1,
    freq_threshold=5,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
)

(1000, 128, 128, 6)


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = torch.load(
    "../weights/SRCNN_best_freq.pth",
    map_location=device
)

model = SRCNN(2, 2)
model = torch.nn.DataParallel(model)
model = model.to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

DataParallel(
  (module): SRCNN(
    (activation): ReLU()
    (conv1): Conv2d(2, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
    (conv2): Conv2d(64, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (conv3): Conv2d(32, 2, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  )
)

In [13]:
results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
)

results

TypeError: list indices must be integers or slices, not str